# Stage 3B on Colab — recover, resume, evaluate

Run the cells in order. The run survives a lost runtime: every checkpoint is
written to Google Drive, and an architecture that was interrupted continues
from the epoch it reached instead of starting again.

**If the runtime dies mid-training**, reconnect and re-run cells 1–3, then the
training cell. `--resume auto` (the default) picks up where the last epoch
left off, restoring the optimiser, scheduler, AMP scaler and RNG state.

**Nothing here touches the official test set** until the final evaluation cell,
which reads it exactly once.

## 1. Mount Drive and fix the persistent paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PERSIST = Path('/content/drive/MyDrive/ensemble-cnn')
CKPT = PERSIST / 'checkpoints'
RESULTS = PERSIST / 'results'
DATA = PERSIST / 'data'
REPO = Path('/content/ensemble-cnn-image-classifier')

for directory in (CKPT, RESULTS, DATA):
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)

## 2. GPU gate

The approved run is CUDA + fp16 at 128×128. On CPU it would take days, so this
cell stops rather than starting something that cannot finish. If it stops,
either switch the runtime type to a T4 or wait for the usage limit to reset —
no checkpoint is lost by waiting.

In [ ]:
import subprocess

import torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
      or 'nvidia-smi is not available on this runtime')

if not torch.cuda.is_available():
    raise SystemExit(
        'No CUDA runtime. Runtime -> Change runtime type -> T4 GPU, or wait for '
        'the Colab usage limit to reset. Do not run Stage 3B on CPU.'
    )

print('torch', torch.__version__, '| cuda', torch.version.cuda)
print('device:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported(), '(T4 is fp16, as configured)')

## 3. Get the repository at the current commit

`git reset --hard` reverts tracked files only. Verified by execution: ignored
paths — `checkpoints/`, `results/`, `data/` — and untracked files survive it, so
this cannot destroy a checkpoint that is sitting in the working clone.

In [ ]:
import os

if not REPO.is_dir():
    !git clone https://github.com/Vamshi044/ensemble-cnn-image-classifier.git {REPO}

os.chdir(REPO)
!git fetch --quiet origin && git checkout --quiet main && git reset --hard origin/main
!git log --oneline -1

# CIFAR-10 lives on Drive so a new runtime does not download it again.
local_data = REPO / 'data'
if not local_data.is_symlink():
    if local_data.is_dir() and any(local_data.iterdir()):
        !cp -rn {local_data}/. {DATA}/
    !rm -rf {local_data}
    !ln -s {DATA} {local_data}
print('data ->', os.path.realpath(local_data))

## 4. Recover anything a previous runtime left behind

Reads every candidate with the project's own loader, reports which architecture
and epoch each one represents, and copies the furthest-along copy of each into
Drive. It never overwrites a Drive file that is already at a later epoch.

In [ ]:
!python3 scripts/recover_checkpoints.py \
    --search /content/ensemble-cnn-image-classifier/checkpoints /content/checkpoints {CKPT} \
    --destination {CKPT} \
    --report {RESULTS}/checkpoint_recovery.json

## 5. Pre-flight

Gates the split checksum, the 45,000/5,000 sizes, the approved hyperparameters,
and that `fit()` has no test-loader parameter. Then it prints the resume plan
for each architecture. Nothing is trained by this cell.

In [ ]:
!python3 scripts/run_stage3b.py --dry-run \
    --checkpoints-dir {CKPT} --results-dir {RESULTS}

## 6. Train / resume

resnet18 → googlenet → vgg11, 10 epochs each, checkpointed to Drive after every
epoch. An architecture already at epoch 10 is skipped, one that was interrupted
continues, one with no checkpoint starts at epoch 1.

Re-running this cell after a disconnect is safe and is the intended recovery
path. It will not overwrite a checkpoint with a fresh start — that requires
`--resume off --force-restart`.

In [ ]:
!python3 scripts/run_stage3b.py \
    --checkpoints-dir {CKPT} --results-dir {RESULTS} \
    2>&1 | tee -a {RESULTS}/stage3b_console.log

## 7. Confirm all three finished before going near the test set

In [ ]:
import json

ready = True
for architecture in ('resnet18', 'googlenet', 'vgg11'):
    path = RESULTS / f'stage3b_{architecture}.json'
    if not path.is_file():
        print(f'{architecture:<10} NO RESULT FILE')
        ready = False
        continue
    run = json.loads(path.read_text())
    best = run.get('best_val_accuracy')
    done = run.get('completed_epochs')
    checkpoints = run.get('checkpoints', {})
    complete = done == 10 and checkpoints.get('best_exists') and checkpoints.get('last_exists')
    ready = ready and bool(complete)
    print(f'{architecture:<10} epochs {done}/10  best val acc {best}  '
          f'split {run.get("split_checksum")}  '
          f'{"OK" if complete else "INCOMPLETE"}')

print('\nready for the ensemble evaluation:', ready)

## 8. Test suite

In [ ]:
!python3 -m pytest -q

## 9. Ensemble evaluation — the one and only read of the test set

Loads each `<arch>_best.pt`, runs the deterministic evaluation transform over
the official 10,000 test images, averages the three softmax distributions with
equal weights, and writes `ensemble_evaluation.json` and
`ensemble_predictions.csv`. Nothing is tuned here.

In [ ]:
!python3 scripts/evaluate_ensemble.py \
    --checkpoints-dir {CKPT} --results-dir {RESULTS}

## 10. Final numbers

In [ ]:
path = RESULTS / 'ensemble_evaluation.json'
if not path.is_file():
    print('No ensemble evaluation yet.')
else:
    report = json.loads(path.read_text())
    for member in report['per_model']:
        print(f"{member['architecture']:<10} "
              f"val {member['validation_accuracy_at_selection']}  "
              f"test {member['test_accuracy']:.4f}")
    print(f"{'ENSEMBLE':<10} test {report['ensemble_test_accuracy']:.4f} "
          f"({report['ensemble_test_correct']}/{report['test_size']})")
    print()
    print('fusion         :', report['fusion'])
    print('weights        :', report['weights'])
    print('split checksum :', report['split_checksum'])
    print('files          :', sorted(f.name for f in RESULTS.iterdir()))
